# AST research

Working notebook for the `ast` research repo. Upstream checkouts are submodules under `refs/`;
our own experiments live in `work/`.

| path | upstream |
|---|---|
| `refs/prism` | https://github.com/ruby/prism — Ruby parser producing the CRuby AST |
| `refs/decode` | https://github.com/socketry/decode — documentation/source analysis built on top of a Ruby parser |
| `refs/rubocop` | https://github.com/rubocop/rubocop — linter whose cops pattern-match over an AST |
| `refs/structuredmerge-ruby` | https://github.com/structuredmerge/structuredmerge-ruby — structured-merge gem monorepo |
| `refs/rdoc` | https://github.com/ruby/rdoc — Ruby's bundled doc generator, with its own parser and code-object model |
| `refs/yard` | https://github.com/lsegal/yard — doc generator with a tag-based object model and its own parsers |
| `refs/ruby-next` | https://github.com/ruby-next/ruby-next — transpiles modern syntax to older Ruby; rewrites trees rather than only reading them |

Note that these do not all speak the same tree: Prism has its own node classes, while RuboCop's cops
are written against `RuboCop::AST` nodes (a `parser`-gem s-expression model, reachable from Prism via
its `Prism::Translation::Parser` compatibility layer). Comparing the two shapes is part of the point.
RDoc, YARD and Decode add a third category — they flatten a tree into a *code-object* model
(classes, methods, docstrings) rather than preserving syntax, so the interesting comparison there is
what each one throws away.

The kernel here is Python; Ruby work is driven through subprocesses so the submodule checkouts
stay the source of truth.

In [12]:
import os, shlex, subprocess, textwrap
from pathlib import Path

REPO = Path.home() / "usb" / "ast"
REFS = REPO / "refs"
WORK = REPO / "work"

PRISM = REFS / "prism"
DECODE = REFS / "decode"
RUBOCOP = REFS / "rubocop"
SMERGE = REFS / "structuredmerge-ruby"
RDOC = REFS / "rdoc"
YARD = REFS / "yard"
RUBY_NEXT = REFS / "ruby-next"

# The kernel inherits a minimal nix PATH, so put the user/system profiles back on it.
os.environ["PATH"] = os.pathsep.join([
    "/etc/profiles/per-user/nathan/bin",
    "/run/current-system/sw/bin",
    str(Path.home() / ".gem-3.4.9" / "bin"),
    os.environ.get("PATH", ""),
])

q = shlex.quote  # repr() escapes newlines; shell single-quoting must not.

def sh(cmd, cwd=REPO):
    p = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    if p.returncode:
        print(f"[exit {p.returncode}] {cmd}")
    return (p.stdout + p.stderr).rstrip()

def ruby(script, *args, cwd=REPO, load_paths=()):
    """Run a (possibly multi-line) Ruby script, passing args through ARGV."""
    inc = " ".join(f"-I{q(str(p))}" for p in load_paths)
    argv = " ".join(q(str(a)) for a in args)
    return sh(f"ruby {inc} -e {q(textwrap.dedent(script))}{' -- ' + argv if argv else ''}", cwd=cwd)

print(REPO, PRISM.is_dir(), DECODE.is_dir())
print(sh("git --version && ruby --version"))
print(sh("git submodule status"))

/home/nathan/usb/ast True True
git version 2.54.0
ruby 3.4.9 (2026-03-11 revision 76cca827ab) +PRISM [x86_64-linux]
 2e3fdd1ae90b7a866b09a1ec98ad32e9d9a71f04 refs/decode (v0.16.0-130-g2e3fdd1)
 6cd6057eacd42ee10315568ba1bfaa181700756f refs/prism (v1.5.2-1093-g6cd6057ea)
 61ca5f84026030957f82406bd8b4d6658dd757ad refs/rdoc (v6.8.1-516-g61ca5f84)
 28d9b78b0e8ae217dfaae934420f059ff2c1ed34 refs/rubocop (v1.92.0-116-g28d9b78b0)
 96201dfc6c42b20dbdb86440c0e425896988d755 refs/ruby-next (v1.0.2-47-g96201df)
 09ebd3ae5a9ae37bccff228d6db584e4ba8cc821 refs/structuredmerge-ruby (v7.1.29-32-g09ebd3ae5)
 3d8670543d2408baea3a1360b2258fde2ee65917 refs/yard (v0.9.45-6-g3d867054)


## Parsing with Prism

Ruby 3.4 ships Prism in the stdlib, so a quick parse needs no build step. Dumping the tree from a
Ruby subprocess keeps the notebook honest about what the real parser produces.

In [16]:
def prism_parse(source, lib=None):
    """Dump the Prism AST for `source`.

    lib=PRISM/'lib' targets the submodule checkout, but that only works once its generated
    files and C extension exist: cd refs/prism && bundle exec rake templates compile
    """
    return ruby(
        'require "prism"; puts Prism.parse(ARGV[0]).value.inspect',
        source,
        load_paths=[lib] if lib else (),
    )

print(prism_parse('def greet(name) = "hi #{name}"'))

@ ProgramNode (location: (1,0)-(1,30))
├── flags: ∅
├── locals: []
└── statements:
    @ StatementsNode (location: (1,0)-(1,30))
    ├── flags: ∅
    └── body: (length: 1)
        └── @ DefNode (location: (1,0)-(1,30))
            ├── flags: newline
            ├── name: :greet
            ├── name_loc: (1,4)-(1,9) = "greet"
            ├── receiver: ∅
            ├── parameters:
            │   @ ParametersNode (location: (1,10)-(1,14))
            │   ├── flags: ∅
            │   ├── requireds: (length: 1)
            │   │   └── @ RequiredParameterNode (location: (1,10)-(1,14))
            │   │       ├── flags: ∅
            │   │       └── name: :name
            │   ├── optionals: (length: 0)
            │   ├── rest: ∅
            │   ├── posts: (length: 0)
            │   ├── keywords: (length: 0)
            │   ├── keyword_rest: ∅
            │   └── block: ∅
            ├── body:
            │   @ StatementsNode (location: (1,18)-(1,30))
            │   ├── flags: ∅
        

## Decode

`decode` is a gem, so it needs its bundle installed in the submodule before it can be driven from
here. Run the next cell once; after that `decode_rb` will work.

In [ ]:
# print(sh("bundle install", cwd=DECODE))

def decode_rb(script, *args):
    """Run a Ruby script with decode's bundle and lib/ loaded."""
    inc = f"-I{q(str(DECODE / 'lib'))}"
    argv = " ".join(q(str(a)) for a in args)
    return sh(f"bundle exec ruby {inc} -e {q(textwrap.dedent(script))}{' -- ' + argv if argv else ''}", cwd=DECODE)

print(decode_rb('require "decode"; puts Decode::VERSION'))

## RuboCop

RuboCop's cops match against `RuboCop::AST` nodes rather than Prism nodes. Needs its bundle
installed in the submodule first.

In [ ]:
# print(sh("bundle install", cwd=RUBOCOP))

def rubocop_ast(source):
    """Dump the RuboCop::AST (parser-gem s-expression) tree for `source`."""
    script = '''
      require "rubocop-ast"
      puts RuboCop::AST::ProcessedSource.new(ARGV[0], RUBY_VERSION.to_f).ast.to_s
    '''
    return sh(f"bundle exec ruby -e {q(textwrap.dedent(script))} -- {q(source)}", cwd=RUBOCOP)

print(rubocop_ast('def greet(name) = "hi #{name}"'))

## StructuredMerge

A gem monorepo under `structuredmerge-ruby/gems`. Two of them are the direct link to the rest of
this repo: `prism-merge` and `ast-crispr-ruby-prism`.

In [15]:
gems = sorted(p.name for p in (SMERGE / "gems").iterdir() if p.is_dir())
print(f"{len(gems)} gems")
for g in gems:
    mark = " <-- prism" if "prism" in g else ""
    print(f"  {g}{mark}")

31 gems
  ast-crispr
  ast-crispr-markdown-markly
  ast-crispr-ruby-prism <-- prism
  ast-merge
  ast-merge-git
  ast-template
  bash-merge
  binary-merge
  citrus-toml-merge
  commonmarker-merge
  dotenv-merge
  go-merge
  html-merge
  json-merge
  kettle-jem
  kramdown-merge
  markdown-merge
  markly-merge
  parslet-toml-merge
  plain-merge
  prism-merge <-- prism
  psych-merge
  rbs-merge
  ruby-merge
  rust-merge
  smorg-rb
  toml-merge
  tree_haver
  typescript-merge
  yaml-merge
  zip-merge


## RDoc and YARD

Both flatten source into a code-object model rather than keeping a syntax tree. RDoc ships with
Ruby, so it needs nothing; YARD runs straight from its submodule `lib/` with no bundle.

In [14]:
SAMPLE = '''# Greets someone.
# @param name [String] who to greet
def greet(name); end
'''

def rdoc_objects(source):
    """Parse `source` with RDoc and report its code-object model."""
    # A bare top-level def lands on Object, not on the file, so walk the store rather than top.
    return ruby('''
        require "rdoc"
        options = RDoc::Options.new
        store = RDoc::Store.new(options)
        top = store.add_file("sample.rb")
        RDoc::Parser::Ruby.new(top, ARGV[0], options, RDoc::Stats.new(store, 1, 0)).scan
        store.complete(:private)
        store.all_classes_and_modules.each do |mod|
          mod.method_list.each do |m|
            puts "#{m.class}  #{mod.full_name}#{m.name_prefix}#{m.name}#{m.params}" \\
                 "  comment=#{m.comment.text.to_s.strip.inspect}"
          end
        end
    ''', source, cwd=RDOC, load_paths=[RDOC / "lib"])

def yard_objects(source):
    """Parse `source` with YARD (from the submodule lib/) and report its object + tag model."""
    return ruby('''
        require "yard"
        YARD::Registry.clear
        YARD.parse_string(ARGV[0])
        YARD::Registry.all.each do |o|
          puts "#{o.class}  #{o.path}  docstring=#{o.docstring.to_s.inspect}"
          o.tags.each do |t|
            puts "    @#{t.tag_name} name=#{t.name.inspect} types=#{t.types.inspect} text=#{t.text.inspect}"
          end
        end
    ''', source, cwd=YARD, load_paths=[YARD / "lib"])

print("--- RDoc ---")
print(rdoc_objects(SAMPLE))
print("\n--- YARD ---")
print(yard_objects(SAMPLE))

--- RDoc ---
RDoc::AnyMethod  Object#greet(name)  comment="Greets someone.\n@param name [String] who to greet"

--- YARD ---
YARD::CodeObjects::MethodObject  #greet  docstring="Greets someone."
    @param name="name" types=["String"] text="who to greet"


## Ruby Next

Ruby Next rewrites modern syntax down to older Ruby, so it is the one submodule here that
*transforms* a tree and prints source back out rather than just reading one. Its rewriters are the
interesting artifact. Needs its bundle installed in the submodule first.

In [ ]:
# print(sh("bundle install", cwd=RUBY_NEXT))

def ruby_next_rewrite(source):
    """Transpile `source` down to older Ruby and show the rewriters that fired."""
    script = '''
      require "ruby-next/language"
      RubyNext::Language.rewriters.each { |r| $stderr.puts "rewriter: #{r}" }
      puts RubyNext::Language.transform(ARGV[0])
    '''
    return sh(f"bundle exec ruby -e {q(textwrap.dedent(script))} -- {q(source)}", cwd=RUBY_NEXT)

print(ruby_next_rewrite('def greet(name) = "hi #{name}"'))

## Our own code

`work/prism_example.rb` walks a file with `Prism::Visitor` and lists the methods it defines. Note
that `node.location.leading_comments` is empty until `result.attach_comments!` runs — Prism keeps
comments out of the tree and only grafts them onto node locations on request.

In [13]:
print(sh(f"ruby {q(str(WORK / 'prism_example.rb'))} {q(str(WORK / 'sample.rb'))}"))

print("\n--- against a file that has doc comments ---")
print(sh(f"ruby {q(str(WORK / 'prism_example.rb'))} {q(str(YARD / 'lib' / 'yard.rb'))}"))

/home/nathan/usb/ast/work/sample.rb: 5 method(s)
  Greeting::Formal#initialize(title) line 5    
  Greeting::Formal#greet(name = DEFAULT) line 9    
  Greeting::Formal.build(title)      line 13   
  Greeting::Casual#greet(name = DEFAULT) line 19   
  (top level)#shout(text)            line 23

--- against a file that has doc comments ---
/home/nathan/usb/ast/refs/yard/lib/yard.rb: 9 method(s)
  YARD.parse(*args)                  line 20   An alias to {Parser::SourceParser}'s parsing method  @example Parse a glob of files YARD.parse('lib/**/*.rb') @see Parser::SourceParser.parse
  YARD.parse_string(*args)           line 27   An alias to {Parser::SourceParser}'s parsing method  @example Parse a string of input YARD.parse_string('class Foo; end') @see Parser::SourceParser.parse_string
  YARD.load_plugins                  line 31   (see YARD::Config.load_plugins) @deprecated Use {Config.load_plugins}
  YARD.windows?                      line 34   @return [Boolean] whether YARD is being run